Silver_to_Gold

Construcao da camada Gold com modelagem dimensional, tabelas ponte, tabela de contexto para IA e consultas analiticas.

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window

spark.sql("CREATE DATABASE IF NOT EXISTS gold")

#Funcao auxiliar para persistir tabelas Gold em formato Delta
def salvar_gold(df, tabela):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela)
    )

#Funcao auxiliar para validar colunas obrigatorias e quantidade de registros
def validar_tabela(nome, colunas_obrigatorias):
    df = spark.table(nome)
    faltantes = [c for c in colunas_obrigatorias if c not in df.columns]
    if faltantes:
        raise RuntimeError(f"{nome}: colunas ausentes: {faltantes}")
    print(f"OK - {nome}: {df.count()} linhas")

Dimensao de filmes

In [0]:
info = spark.table("silver.tb_info_filmes")

#A chave substituta e gerada de forma deterministica pela chave natural do filme
w_movies = Window.orderBy(f.col("id_filme"))

dim_movies = (
    info.select(
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
    .dropDuplicates(["id_filme"])
    .withColumn("sk_movie_id", f.row_number().over(w_movies).cast("bigint"))
    .select(
        "sk_movie_id",
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
)

salvar_gold(dim_movies, "gold.dim_movies")

dim_movies.orderBy("sk_movie_id").show(10, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+--------+------------------------------------------+---------------+--------------+---------------+---------------+------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Dimensao de generos

In [0]:
generos = spark.table("silver.tb_generos")

w_genres = Window.orderBy(f.col("nome_genero"))

dim_genres = (
    generos.select("nome_genero")
    .filter(f.col("nome_genero").isNotNull())
    .dropDuplicates(["nome_genero"])
    .withColumn("sk_genre_id", f.row_number().over(w_genres).cast("bigint"))
    .select("sk_genre_id", "nome_genero")
)

salvar_gold(dim_genres, "gold.dim_genres")

dim_genres.orderBy("sk_genre_id").show(50, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+---------------+
|sk_genre_id|nome_genero    |
+-----------+---------------+
|1          |Action         |
|2          |Adventure      |
|3          |Animation      |
|4          |Comedy         |
|5          |Crime          |
|6          |Documentary    |
|7          |Drama          |
|8          |Family         |
|9          |Fantasy        |
|10         |History        |
|11         |Horror         |
|12         |Music          |
|13         |Mystery        |
|14         |Romance        |
|15         |Science Fiction|
|16         |TV Movie       |
|17         |Thriller       |
|18         |War            |
|19         |Western        |
+-----------+---------------+



Dimensoes de pessoas e produtoras

In [0]:
entidades = spark.table("silver.tb_pessoas_empresas")

#A dimensao de pessoas contempla somente Ator, Diretor e Roteirista
pessoas = (
    entidades
    .filter(f.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        f.col("nome_entidade").alias("nome_pessoa"),
        f.col("tipo_entidade").alias("tipo_pessoa")
    )
    .filter(f.col("nome_pessoa").isNotNull())
    .dropDuplicates(["nome_pessoa", "tipo_pessoa"])
)

w_people = Window.orderBy(f.col("nome_pessoa"), f.col("tipo_pessoa"))

dim_people = (
    pessoas
    .withColumn("sk_person_id", f.row_number().over(w_people).cast("bigint"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

salvar_gold(dim_people, "gold.dim_people")

#A dimensao de empresas contempla somente produtoras
empresas = (
    entidades
    .filter(f.col("tipo_entidade") == "Produtora")
    .select(f.col("nome_entidade").alias("nome_produtora"))
    .filter(f.col("nome_produtora").isNotNull())
    .dropDuplicates(["nome_produtora"])
)

w_companies = Window.orderBy(f.col("nome_produtora"))

dim_companies = (
    empresas
    .withColumn("sk_company_id", f.row_number().over(w_companies).cast("bigint"))
    .select("sk_company_id", "nome_produtora")
)

salvar_gold(dim_companies, "gold.dim_companies")

dim_people.orderBy("sk_person_id").show(10, truncate=False)
dim_companies.orderBy("sk_company_id").show(10, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------------+------------------------+-----------+
|sk_person_id|nome_pessoa             |tipo_pessoa|
+------------+------------------------+-----------+
|1           |'ana Ika                |Ator       |
|2           |'e-gotti' Eric Johnson  |Ator       |
|3           |'jeeva' Ravi            |Ator       |
|4           |'meesai' Mohan          |Ator       |
|5           |'meesai' Rajendran      |Ator       |
|6           |'om' Rakesh Chaturvedi  |Ator       |
|7           |'poo' Ram               |Ator       |
|8           |'sunday Jeff' Silverman |Ator       |
|9           |'weird Al' Yankovic     |Roteirista |
|10          |'wáats'asdíyei Joe Yates|Diretor    |
+------------+------------------------+-----------+
only showing top 10 rows
+-------------+-----------------------------------+
|sk_company_id|nome_produtora                     |
+-------------+-----------------------------------+
|1            |#1nfluence Production              |
|2            |#beardforce Films       

Tabelas ponte

In [0]:
#Bridge entre filmes e generos
bridge_movie_genre = (
    spark.table("silver.tb_generos").alias("g")
    .join(
        dim_movies.select("sk_movie_id", "id_filme").alias("m"),
        f.col("g.id_filme") == f.col("m.id_filme"),
        "inner"
    )
    .join(
        dim_genres.alias("d"),
        f.col("g.nome_genero") == f.col("d.nome_genero"),
        "inner"
    )
    .select("sk_movie_id", "sk_genre_id")
    .dropDuplicates()
)

salvar_gold(bridge_movie_genre, "gold.bridge_movie_genre")

#Bridge entre filmes e pessoas
pessoas_silver = (
    spark.table("silver.tb_pessoas_empresas")
    .filter(f.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
)

bridge_movie_person = (
    pessoas_silver.alias("p")
    .join(
        dim_movies.select("sk_movie_id", "id_filme").alias("m"),
        f.col("p.id_filme") == f.col("m.id_filme"),
        "inner"
    )
    .join(
        dim_people.alias("d"),
        (f.col("p.nome_entidade") == f.col("d.nome_pessoa"))
        & (f.col("p.tipo_entidade") == f.col("d.tipo_pessoa")),
        "inner"
    )
    .select("sk_movie_id", "sk_person_id")
    .dropDuplicates()
)

salvar_gold(bridge_movie_person, "gold.bridge_movie_person")

#Bridge entre filmes e produtoras
empresas_silver = (
    spark.table("silver.tb_pessoas_empresas")
    .filter(f.col("tipo_entidade") == "Produtora")
)

bridge_movie_company = (
    empresas_silver.alias("e")
    .join(
        dim_movies.select("sk_movie_id", "id_filme").alias("m"),
        f.col("e.id_filme") == f.col("m.id_filme"),
        "inner"
    )
    .join(
        dim_companies.alias("d"),
        f.col("e.nome_entidade") == f.col("d.nome_produtora"),
        "inner"
    )
    .select("sk_movie_id", "sk_company_id")
    .dropDuplicates()
)

salvar_gold(bridge_movie_company, "gold.bridge_movie_company")

bridge_movie_genre.show(10, truncate=False)
bridge_movie_person.show(10, truncate=False)
bridge_movie_company.show(10, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-----------+
|sk_movie_id|sk_genre_id|
+-----------+-----------+
|44         |7          |
|55         |8          |
|58         |14         |
|68         |17         |
|85         |7          |
|96         |4          |
|110        |11         |
|117        |17         |
|118        |5          |
|122        |9          |
+-----------+-----------+
only showing top 10 rows
+-----------+------------+
|sk_movie_id|sk_person_id|
+-----------+------------+
|30093      |202549      |
|16308      |79716       |
|26736      |120803      |
|41         |256694      |
|62138      |350267      |
|6293       |251580      |
|23561      |238660      |
|25554      |167676      |
|58742      |8188        |
|76883      |238653      |
+-----------+------------+
only showing top 10 rows
+-----------+-------------+
|sk_movie_id|sk_company_id|
+-----------+-------------+
|2          |20893        |
|4          |28954        |
|5          |15897        |
|6          |9201         |
|8          

Dimensao de avaliacoes

In [0]:
avaliacoes = spark.table("silver.tb_avaliacoes_usuarios")

#As avaliacoes sao resumidas por filme conforme o grao definido para a dimensao
reviews_resumo = (
    avaliacoes
    .groupBy("id_filme")
    .agg(
        f.count(f.lit(1)).cast("int").alias("qtd_avaliacoes_usuarios"),
        f.round(f.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
    )
)

w_reviews = Window.orderBy(f.col("sk_movie_id"))

dim_reviews = (
    reviews_resumo.alias("r")
    .join(
        dim_movies.select("sk_movie_id", "id_filme").alias("m"),
        f.col("r.id_filme") == f.col("m.id_filme"),
        "inner"
    )
    .select(
        f.col("m.sk_movie_id").alias("sk_movie_id"),
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
    .withColumn("sk_review_id", f.row_number().over(w_reviews).cast("bigint"))
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)

salvar_gold(dim_reviews, "gold.dim_reviews")

dim_reviews.orderBy("sk_review_id").show(10, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------------+-----------+-----------------------+-------------------+
|sk_review_id|sk_movie_id|qtd_avaliacoes_usuarios|nota_media_usuarios|
+------------+-----------+-----------------------+-------------------+
|1           |1          |1                      |0.1                |
|2           |5          |1                      |3.3                |
|3           |8          |1                      |6.8                |
|4           |17         |1                      |5.4                |
|5           |19         |1                      |5.3                |
|6           |20         |1                      |4.4                |
|7           |23         |3                      |4.75               |
|8           |24         |1                      |4.8                |
|9           |26         |1                      |6.6                |
|10          |33         |1                      |7.5                |
+------------+-----------+-----------------------+-------------------+
only s

Tabela fato de performance

In [0]:
financeiro = spark.table("silver.tb_financeiro_filmes")
metricas = spark.table("silver.tb_metricas_engajamento")

#A fato possui um unico registro por filme lancado e preserva o grao durante os joins
filmes_lancados = (
    dim_movies
    .filter(f.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
)

fact_movies_performance = (
    filmes_lancados.alias("m")
    .join(
        financeiro.alias("f"),
        f.col("m.id_filme") == f.col("f.id_filme"),
        "left"
    )
    .join(
        metricas.alias("e"),
        f.col("m.id_filme") == f.col("e.id_filme"),
        "left"
    )
    .select(
        f.col("m.sk_movie_id").cast("bigint").alias("sk_movie_id"),
        f.col("f.orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
        f.col("f.receita_usd").cast("decimal(18,2)").alias("receita_usd"),
        f.col("f.lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),
        f.col("f.orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
        f.col("f.receita_brl").cast("decimal(18,2)").alias("receita_brl"),
        f.col("f.lucro_brl").cast("decimal(18,2)").alias("lucro_brl"),
        f.col("e.popularidade").cast("double").alias("popularidade"),
        f.col("e.nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        f.col("e.qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
        f.col("e.nota_media_imdb").cast("double").alias("nota_media_imdb"),
        f.col("e.qtd_votos_imdb").cast("int").alias("qtd_votos_imdb")
    )
)

salvar_gold(fact_movies_performance, "gold.fact_movies_performance")

fact_movies_performance.printSchema()
fact_movies_performance.show(10, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


root
 |-- sk_movie_id: long (nullable = false)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- popularidade: double (nullable = true)
 |-- nota_media_tmdb: double (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: double (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)

+-----------+-------------+-----------+---------+-------------+-----------+---------+------------+---------------+--------------+---------------+--------------+
|sk_movie_id|orcamento_usd|receita_usd|lucro_usd|orcamento_brl|receita_brl|lucro_brl|popularidade|nota_media_tmdb|qtd_votos_tmdb|nota_media_imdb|qtd_votos_imdb|
+-----------+-------------+-----------+---------+-------------+-----------+---------+------------+

Tabela de contexto para IA

In [0]:
#Agregacao de atores por filme para formar uma unica string de contexto
atores_por_filme = (
    bridge_movie_person.alias("b")
    .join(dim_people.alias("p"), "sk_person_id", "inner")
    .filter(f.col("p.tipo_pessoa") == "Ator")
    .groupBy("sk_movie_id")
    .agg(
        f.concat_ws(
            ", ",
            f.sort_array(f.collect_set("nome_pessoa"))
        ).alias("atores")
    )
)

#Agregacao de diretores por filme para evitar perda de informacao em obras com mais de um diretor
diretores_por_filme = (
    bridge_movie_person.alias("b")
    .join(dim_people.alias("p"), "sk_person_id", "inner")
    .filter(f.col("p.tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(
        f.concat_ws(
            ", ",
            f.sort_array(f.collect_set("nome_pessoa"))
        ).alias("diretores")
    )
)

contexto_base = (
    fact_movies_performance.alias("f")
    .join(dim_movies.alias("m"), "sk_movie_id", "inner")
    .join(atores_por_filme.alias("a"), "sk_movie_id", "left")
    .join(diretores_por_filme.alias("d"), "sk_movie_id", "left")
)

#Coalesce garante que campos nulos nao transformem o documento inteiro em NULL
receita_txt = f.when(
    f.col("f.receita_usd").isNotNull(),
    f.concat(f.lit("US$ "), f.format_number(f.col("f.receita_usd"), 2))
).otherwise(f.lit("valor não informado"))

orcamento_txt = f.when(
    f.col("f.orcamento_usd").isNotNull(),
    f.concat(f.lit("US$ "), f.format_number(f.col("f.orcamento_usd"), 2))
).otherwise(f.lit("valor não informado"))

ano_txt = f.coalesce(
    f.col("m.ano_lancamento").cast("string"),
    f.lit("ano não informado")
)

atores_txt = f.when(
    f.col("a.atores").isNotNull() & (f.trim(f.col("a.atores")) != ""),
    f.col("a.atores")
).otherwise(f.lit("elenco não informado"))

diretores_txt = f.when(
    f.col("d.diretores").isNotNull() & (f.trim(f.col("d.diretores")) != ""),
    f.col("d.diretores")
).otherwise(f.lit("direção não informada"))

sinopse_txt = f.when(
    f.col("m.sinopse").isNotNull() & (f.trim(f.col("m.sinopse")) != ""),
    f.trim(f.col("m.sinopse"))
).otherwise(f.lit("sinopse não informada"))

gold_genai_movies_context = (
    contexto_base
    .select(
        f.col("m.id_filme").alias("movie_id"),
        f.col("m.titulo").alias("title"),
        f.concat(
            f.lit("O filme "),
            f.coalesce(f.col("m.titulo"), f.lit("título não informado")),
            f.lit(", lancado no ano de "),
            ano_txt,
            f.lit(", faturou "),
            receita_txt,
            f.lit(" e teve um custo de "),
            orcamento_txt,
            f.lit(". Estrelado por "),
            atores_txt,
            f.lit(" e dirigido por "),
            diretores_txt,
            f.lit(", o filme possui a seguinte sinopse: "),
            sinopse_txt,
            f.lit(".")
        ).alias("llm_context_document")
    )
)

salvar_gold(gold_genai_movies_context, "gold.gold_genai_movies_context")

gold_genai_movies_context.show(10, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------+------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Validacoes da modelagem Gold

In [0]:
validar_tabela("gold.fact_movies_performance", [
    "sk_movie_id", "orcamento_usd", "receita_usd", "lucro_usd",
    "orcamento_brl", "receita_brl", "lucro_brl", "popularidade",
    "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"
])

validar_tabela("gold.dim_movies", [
    "sk_movie_id", "id_filme", "titulo", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "idioma_original", "status_filme", "sinopse"
])

validar_tabela("gold.dim_genres", ["sk_genre_id", "nome_genero"])
validar_tabela("gold.dim_people", ["sk_person_id", "nome_pessoa", "tipo_pessoa"])
validar_tabela("gold.dim_companies", ["sk_company_id", "nome_produtora"])

validar_tabela("gold.dim_reviews", [
    "sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios"
])

validar_tabela("gold.bridge_movie_genre", ["sk_movie_id", "sk_genre_id"])
validar_tabela("gold.bridge_movie_person", ["sk_movie_id", "sk_person_id"])
validar_tabela("gold.bridge_movie_company", ["sk_movie_id", "sk_company_id"])

validar_tabela("gold.gold_genai_movies_context", [
    "movie_id", "title", "llm_context_document"
])

#Validacao de unicidade das chaves primarias e do grao da tabela fato
assert dim_movies.groupBy("sk_movie_id").count().filter(f.col("count") > 1).count() == 0
assert dim_movies.groupBy("id_filme").count().filter(f.col("count") > 1).count() == 0
assert dim_genres.groupBy("sk_genre_id").count().filter(f.col("count") > 1).count() == 0
assert dim_people.groupBy("sk_person_id").count().filter(f.col("count") > 1).count() == 0
assert dim_companies.groupBy("sk_company_id").count().filter(f.col("count") > 1).count() == 0
assert dim_reviews.groupBy("sk_review_id").count().filter(f.col("count") > 1).count() == 0
assert fact_movies_performance.groupBy("sk_movie_id").count().filter(f.col("count") > 1).count() == 0

#Validacao de duplicidade nas tabelas ponte
assert bridge_movie_genre.groupBy("sk_movie_id", "sk_genre_id").count().filter(f.col("count") > 1).count() == 0
assert bridge_movie_person.groupBy("sk_movie_id", "sk_person_id").count().filter(f.col("count") > 1).count() == 0
assert bridge_movie_company.groupBy("sk_movie_id", "sk_company_id").count().filter(f.col("count") > 1).count() == 0

#Validacao da tabela de contexto para garantir documentos nao nulos
assert gold_genai_movies_context.filter(
    f.col("movie_id").isNull()
    | f.col("llm_context_document").isNull()
    | (f.trim("llm_context_document") == "")
).count() == 0

print("Modelagem Gold validada com sucesso.")

OK - gold.fact_movies_performance: 96463 linhas
OK - gold.dim_movies: 97879 linhas
OK - gold.dim_genres: 19 linhas
OK - gold.dim_people: 418712 linhas
OK - gold.dim_companies: 45101 linhas
OK - gold.dim_reviews: 27303 linhas
OK - gold.bridge_movie_genre: 140426 linhas
OK - gold.bridge_movie_person: 757819 linhas
OK - gold.bridge_movie_company: 116205 linhas
OK - gold.gold_genai_movies_context: 96463 linhas


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Modelagem Gold validada com sucesso.


Consulta 1 - Receita total em BRL

In [0]:
q1_receita_total = (
    spark.table("gold.fact_movies_performance")
    .agg(
        f.sum("receita_brl").cast("decimal(20,2)").alias("receita_total_brl")
    )
)

display(q1_receita_total)

receita_total_brl
834732290730.20


Consulta 2 - Cinco filmes com maior popularidade

In [0]:
q2_top_popularidade = (
    spark.table("gold.fact_movies_performance").alias("f")
    .join(
        spark.table("gold.dim_movies").alias("m"),
        "sk_movie_id",
        "inner"
    )
    .filter(f.col("f.popularidade").isNotNull())
    .select(
        f.col("m.titulo").alias("titulo"),
        f.col("f.popularidade").alias("popularidade")
    )
    .orderBy(f.col("popularidade").desc(), f.col("titulo"))
    .limit(5)
)

display(q2_top_popularidade)

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
wwe survivor series 2018,2018.0


Consulta 3 - Quantidade de filmes por genero

In [0]:
q3_filmes_por_genero = (
    spark.table("gold.bridge_movie_genre").alias("b")
    .join(
        spark.table("gold.dim_genres").alias("g"),
        "sk_genre_id",
        "inner"
    )
    .groupBy(f.col("g.nome_genero"))
    .agg(
        f.countDistinct("sk_movie_id").alias("qtd_filmes")
    )
    .orderBy(f.col("qtd_filmes").desc(), f.col("nome_genero"))
)

display(q3_filmes_por_genero)

nome_genero,qtd_filmes
Drama,32286
Documentary,18996
Comedy,18624
Thriller,10274
Horror,9729
Romance,7639
Action,6049
Crime,4747
Animation,4469
TV Movie,4079


Consulta 4 - Ranking dos dez filmes com maior receita

In [0]:
base_receita = (
    spark.table("gold.fact_movies_performance").alias("f")
    .join(
        spark.table("gold.dim_movies").alias("m"),
        "sk_movie_id",
        "inner"
    )
    .filter(f.col("f.receita_usd").isNotNull())
    .select(
        f.col("m.titulo").alias("titulo"),
        f.col("f.receita_usd").alias("receita_usd"),
        f.col("f.receita_brl").alias("receita_brl")
    )
)

w_receita = Window.orderBy(f.col("receita_usd").desc())

q4_ranking_receita = (
    base_receita
    .withColumn("posicao_ranking", f.rank().over(w_receita))
    .orderBy("posicao_ranking", "titulo")
    .limit(10)
)

display(q4_ranking_receita)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


titulo,receita_usd,receita_brl,posicao_ranking
Avengers: Endgame,2800000000.00,14439320000.00,1
Avatar: The Way of Water,2320250281.00,11965298674.09,2
AVENGERS: INFINITY WAR,2052415039.00,10584099114.62,3
spider-man: no way home,1921847111.00,9910773366.72,4
The Lion King,1663075401.00,8576313535.42,5
Top Gun: Maverick,1488732821.00,7677246284.61,6
Barbie,1428545028.00,7366863854.89,7
The Super Mario Bros. Movie,1355725263.00,6991339608.76,8
Black Panther,1349926083.00,6961433817.42,9
Star Wars: The Last Jedi,1332698830.00,6872594596.43,10


Consulta 5 - Ator com maior quantidade de participacoes nos ultimos dois anos

In [0]:
#A data de referencia e a data de lancamento mais recente entre filmes lancados e nao futuros
data_referencia = (
    spark.table("gold.dim_movies")
    .filter(
        (f.col("status_filme") == "Lançado")
        & f.col("data_lancamento").isNotNull()
        & (f.col("data_lancamento") <= f.current_date())
    )
    .agg(f.max("data_lancamento").alias("data_maxima"))
)

limites_2_anos = data_referencia.select(
    f.col("data_maxima"),
    f.add_months(f.col("data_maxima"), -24).alias("data_inicio")
)

filmes_2_anos = (
    spark.table("gold.dim_movies").alias("m")
    .crossJoin(limites_2_anos.alias("l"))
    .filter(
        (f.col("m.status_filme") == "Lançado")
        & f.col("m.data_lancamento").between(
            f.col("l.data_inicio"),
            f.col("l.data_maxima")
        )
    )
    .select("sk_movie_id")
)

atores_2_anos = (
    spark.table("gold.bridge_movie_person").alias("b")
    .join(filmes_2_anos.alias("m"), "sk_movie_id", "inner")
    .join(
        spark.table("gold.dim_people").alias("p"),
        "sk_person_id",
        "inner"
    )
    .filter(f.col("p.tipo_pessoa") == "Ator")
    .groupBy("sk_person_id", "nome_pessoa")
    .agg(
        f.countDistinct("sk_movie_id").alias("qtd_participacoes")
    )
)

w_ator = Window.orderBy(f.col("qtd_participacoes").desc())

q5_ator = (
    atores_2_anos
    .withColumn("posicao", f.rank().over(w_ator))
    .filter(f.col("posicao") == 1)
    .select("nome_pessoa", "qtd_participacoes")
)

display(q5_ator)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


nome_pessoa,qtd_participacoes
John Simmonds,12
Adlih Torres,12
Jonnathon Cripple,12
Sterling L. Pope,12
Andy Dubitsky,12
Gloria Karel,12
Tristan Welsh,12
Cooper Tomlinson,12


Consulta 6 - Produtora com maior lucro nos ultimos cinco anos

In [0]:
#O mesmo criterio de data limite superior e aplicado ao recorte de cinco anos
limites_5_anos = data_referencia.select(
    f.col("data_maxima"),
    f.add_months(f.col("data_maxima"), -60).alias("data_inicio")
)

filmes_5_anos = (
    spark.table("gold.dim_movies").alias("m")
    .crossJoin(limites_5_anos.alias("l"))
    .filter(
        (f.col("m.status_filme") == "Lançado")
        & f.col("m.data_lancamento").between(
            f.col("l.data_inicio"),
            f.col("l.data_maxima")
        )
    )
    .select("sk_movie_id")
)

lucro_produtoras = (
    spark.table("gold.bridge_movie_company").alias("b")
    .join(filmes_5_anos.alias("m"), "sk_movie_id", "inner")
    .join(
        spark.table("gold.fact_movies_performance")
        .select("sk_movie_id", "lucro_brl")
        .alias("f"),
        "sk_movie_id",
        "inner"
    )
    .join(
        spark.table("gold.dim_companies").alias("c"),
        "sk_company_id",
        "inner"
    )
    .filter(f.col("f.lucro_brl").isNotNull())
    .groupBy("sk_company_id", "nome_produtora")
    .agg(
        f.sum("lucro_brl").cast("decimal(20,2)").alias("lucro_total_brl")
    )
)

w_produtora = Window.orderBy(f.col("lucro_total_brl").desc())

q6_produtora = (
    lucro_produtoras
    .withColumn("posicao", f.rank().over(w_produtora))
    .filter(f.col("posicao") == 1)
    .select("nome_produtora", "lucro_total_brl")
)

display(q6_produtora)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


nome_produtora,lucro_total_brl
Universal Pictures,28111962021.64


Conclusao

In [0]:
print("Silver_to_Gold concluido com sucesso.")

Silver_to_Gold concluido com sucesso.
